# Map California's largest emitters

**Terrain advanced · Python project · Climate TRACE**

Load facility-level greenhouse gas estimates for California, rank the largest sources, map where they cluster, and break emissions down by sector.

---

## Learning objectives

By the end, you should be able to:

- Work with a **facility-level emissions inventory** (not country totals)
- Rank and aggregate by sector and location
- Map point sources with size or color tied to emissions
- Describe TRACE as a **model-based screening tool**, not legal compliance data

**Time:** ~3 hr · **Dataset:** [Climate TRACE explainer](../explainer.html?id=climate-trace)

---

## For mentors

- **Prerequisites:** One intermediate project; comfort with pandas and basic maps.
- **Data:** Real **Climate TRACE v5.7.0 CO₂** facility data for California (2022). Default loads a verified extract; set `REFRESH_FROM_CLIMATE_TRACE = True` to re-download from [downloads.climatetrace.org](https://downloads.climatetrace.org/latest/country_packages/co2/USA.zip) (~300 MB).
- **Watch for:** Students treating estimates as exact tonnage — emphasize uncertainty and methodology.

**Suggested flow:** Read each section → student runs the cell → use *Mentor check-in* before continuing.

---

**Goal:** Load libraries and choose your data source.

**Mentor check-in:** *What's the difference between a national inventory total and a facility-level row?*

In [ ]:
from pathlib import Path
import zipfile
import io
import pandas as pd
import matplotlib.pyplot as plt
import folium
from folium.plugins import MarkerCluster
import requests

REFRESH_FROM_CLIMATE_TRACE = False  # True = re-download official USA CO2 zip (~300 MB, ~5 min)
REPORT_YEAR = 2022
TOP_N = 15
MAP_CENTER = [36.8, -119.5]
CLIMATE_TRACE_ZIP = "https://downloads.climatetrace.org/latest/country_packages/co2/USA.zip"
SECTOR_FILES = [
    "DATA/power/electricity-generation_emissions_sources_v5_7_0.csv",
    "DATA/fossil_fuel_operations/oil-and-gas-refining_emissions_sources_v5_7_0.csv",
    "DATA/fossil_fuel_operations/oil-and-gas-production_emissions_sources_v5_7_0.csv",
    "DATA/manufacturing/cement_emissions_sources_v5_7_0.csv",
]
GITHUB_RAW = "https://raw.githubusercontent.com/KrishM23/Environmental-Data-Science/main/notebooks/data"

**Goal:** Import the facility table from Climate TRACE.

**Concept:** Each row is one **asset** (power plant, refinery, etc.) with lat/lon and annual CO₂ (metric tonnes) for `REPORT_YEAR`.

**You should see:** ~330+ California facilities from Climate TRACE v5.7.0.

**If something breaks:** Keep `REFRESH_FROM_CLIMATE_TRACE = False` to use the bundled real extract from GitHub.

In [ ]:
def filter_ca_facilities(raw):
    raw = raw.copy()
    raw["lat"] = pd.to_numeric(raw["lat"], errors="coerce")
    raw["lon"] = pd.to_numeric(raw["lon"], errors="coerce")
    raw["emissions_quantity"] = pd.to_numeric(raw["emissions_quantity"], errors="coerce")
    raw["start_time"] = pd.to_datetime(raw["start_time"], errors="coerce")
    raw["year"] = raw["start_time"].dt.year
    ca = raw[
        raw["lat"].between(32.5, 42)
        & raw["lon"].between(-124.5, -114)
        & (raw["year"] == REPORT_YEAR)
    ]
    annual = ca.groupby(
        ["source_id", "source_name", "sector", "subsector", "lat", "lon", "gas"],
        as_index=False,
    )["emissions_quantity"].sum()
    annual = annual.rename(columns={"emissions_quantity": "emissions_co2e_tonnes"})
    annual["year"] = REPORT_YEAR
    return annual.sort_values("emissions_co2e_tonnes", ascending=False)

def load_from_climate_trace_zip():
    print("Downloading Climate TRACE USA CO2 package (~300 MB)...")
    resp = requests.get(CLIMATE_TRACE_ZIP, timeout=600, stream=True)
    resp.raise_for_status()
    buf = io.BytesIO(resp.content)
    cols = ["source_id", "source_name", "sector", "subsector", "start_time", "lat", "lon", "gas", "emissions_quantity"]
    frames = []
    with zipfile.ZipFile(buf) as zf:
        for name in SECTOR_FILES:
            print(f"  Reading {name.split('/')[-1]}")
            frames.append(pd.read_csv(zf.open(name), usecols=cols, low_memory=False))
    combined = pd.concat(frames, ignore_index=True)
    return filter_ca_facilities(combined), "Climate TRACE CO2 USA zip (live download)"

def load_bundled_extract():
    for path in [Path("data/climate_trace_ca_facilities.csv"), Path("notebooks/data/climate_trace_ca_facilities.csv")]:
        if path.exists():
            return pd.read_csv(path), str(path)
    import urllib.request
    from io import StringIO
    url = f"{GITHUB_RAW}/climate_trace_ca_facilities.csv"
    with urllib.request.urlopen(url, timeout=60) as resp:
        return pd.read_csv(StringIO(resp.read().decode())), url

if REFRESH_FROM_CLIMATE_TRACE:
    facilities, source = load_from_climate_trace_zip()
else:
    facilities, source = load_bundled_extract()

print(f"Loaded {len(facilities):,} California facilities from {source}")
facilities.head()

**Goal:** Rank the top emitters and summarize by sector.

**You should see:** Natural-gas power plants dominate the top of the list; sector totals show where inventory mass concentrates.

**Mentor check-in:** *Why might a refinery rank lower than a gas plant but still matter locally for air quality?*

In [ ]:
top = facilities.head(TOP_N).copy()
top["emissions_mt"] = top["emissions_co2e_tonnes"] / 1e6

sector_totals = (
    facilities.groupby("sector", as_index=False)["emissions_co2e_tonnes"]
    .sum()
    .assign(emissions_mt=lambda d: d["emissions_co2e_tonnes"] / 1e6)
    .sort_values("emissions_co2e_tonnes", ascending=False)
)

print("Top facilities:")
display_cols = ["source_name", "sector", "subsector", "emissions_mt"]
print(top[display_cols].to_string(index=False, formatters={"emissions_mt": "{:.2f}".format}))
print("\nSector totals:")
print(sector_totals[["sector", "emissions_mt"]].to_string(index=False, formatters={"emissions_mt": "{:.2f}".format}))

**Goal:** Bar chart of the top facilities.

**You should see:** Horizontal bars labeled with plant names.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top["source_name"][::-1], top["emissions_mt"][::-1], color="#2F6B4F")
ax.set_xlabel("Emissions (million metric tons CO₂)")
ax.set_title(f"Top {TOP_N} emitters in California (Climate TRACE CO₂, {REPORT_YEAR})")
plt.tight_layout()
plt.show()

**Goal:** Map facilities — marker size scales with emissions.

**You should see:** Clusters near LA, Bay Area, and Central Valley power plants.

**Mentor check-in:** *Do the largest dots fall where you expected California's energy infrastructure to be?*

In [ ]:
max_em = facilities["emissions_co2e_tonnes"].max()
m = folium.Map(location=MAP_CENTER, zoom_start=6, tiles="CartoDB positron")
cluster = MarkerCluster().add_to(m)

for _, row in facilities.iterrows():
    radius = 4 + 20 * (row["emissions_co2e_tonnes"] / max_em)
    mt = row["emissions_co2e_tonnes"] / 1e6
    popup = folium.Popup(
        f"<b>{row['source_name']}</b><br>"
        f"Sector: {row['sector']} / {row['subsector']}<br>"
        f"{mt:.2f} Mt CO₂ ({REPORT_YEAR})",
        max_width=280,
    )
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=radius,
        popup=popup,
        color="#7B2D26",
        fill=True,
        fill_color="#D98E5A",
        fill_opacity=0.75,
    ).add_to(cluster)

m

**Goal:** Sector breakdown chart.

**You should see:** Which sectors dominate total emissions in this sample.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(sector_totals["sector"], sector_totals["emissions_mt"], color="#5B8BB0")
ax.set_ylabel("Million metric tons CO₂")
ax.set_title(f"Emissions by sector — California ({REPORT_YEAR})")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

---

## Final takeaway

**Mentor prompts:**

1. Name the **#1 facility** and its sector — why is it large?
2. Where do emitters **cluster** geographically?
3. Why is Climate TRACE useful for accountability even if it is not a legal inventory?

**Mentor rubric:** Uses facility-level language · Names a limitation · Avoids overclaiming precision

**Extension:** Download the full USA power-sector CSV from [climatetrace.org/data](https://climatetrace.org/data), set `CUSTOM_CSV_PATH`, and re-run.

**Next:** [Climate TRACE explainer](../explainer.html?id=climate-trace) · [Burden + emitters (R project)](../projects.html)